# World Models & Video Diffusion Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: 3D patchify for video

In [ ]:
```python

import torch

import torch.nn as nn

class VideoPatch3D(nn.Module):

    def __init__(self, in_channels=4, dim=64, patch_t=2, patch_h=2, patch_w=2):

        super().__init__()

        self.proj = nn.Conv3d(

            in_channels, dim,

            kernel_size=(patch_t, patch_h, patch_w),

            stride=(patch_t, patch_h, patch_w),

        )

        self.patch_t = patch_t

        self.patch_h = patch_h

        self.patch_w = patch_w

    def forward(self, x):

        # x: (N, C, T, H, W)

        x = self.proj(x)

        n, c, t, h, w = x.shape

        tokens = x.reshape(n, c, t * h * w).transpose(1, 2)

        return tokens, (t, h, w)

In [ ]:
```

A 3D conv with stride equal to kernel acts as the spatio-temporal patchifier. `(T, H, W) -> (T/2, H/2, W/2)` grid of tokens.

### Step 2: 3D rotary position encoding

Rotary Position Embeddings (RoPE) separately applied along `t`, `h`, `w` axes:

In [ ]:
```python

def rope_3d(tokens, t_dim, h_dim, w_dim, grid):

    """

    tokens: (N, T*H*W, D)

    grid: (T, H, W) sizes

    t_dim + h_dim + w_dim == D

    """

    T, H, W = grid

    n, seq, d = tokens.shape

    if t_dim + h_dim + w_dim != d:

        raise ValueError(f"t_dim+h_dim+w_dim ({t_dim}+{h_dim}+{w_dim}) must equal D={d}")

    assert seq == T * H * W

    t_idx = torch.arange(T, device=tokens.device).repeat_interleave(H * W)

    h_idx = torch.arange(H, device=tokens.device).repeat_interleave(W).repeat(T)

    w_idx = torch.arange(W, device=tokens.device).repeat(T * H)

    # Simplified: just scale channels by frequencies. Real RoPE rotates pairs.

    freqs_t = torch.exp(-torch.log(torch.tensor(10000.0)) * torch.arange(t_dim // 2, device=tokens.device) / (t_dim // 2))

    freqs_h = torch.exp(-torch.log(torch.tensor(10000.0)) * torch.arange(h_dim // 2, device=tokens.device) / (h_dim // 2))

    freqs_w = torch.exp(-torch.log(torch.tensor(10000.0)) * torch.arange(w_dim // 2, device=tokens.device) / (w_dim // 2))

    emb_t = torch.cat([torch.sin(t_idx[:, None] * freqs_t), torch.cos(t_idx[:, None] * freqs_t)], dim=-1)

    emb_h = torch.cat([torch.sin(h_idx[:, None] * freqs_h), torch.cos(h_idx[:, None] * freqs_h)], dim=-1)

    emb_w = torch.cat([torch.sin(w_idx[:, None] * freqs_w), torch.cos(w_idx[:, None] * freqs_w)], dim=-1)

    return tokens + torch.cat([emb_t, emb_h, emb_w], dim=-1)

In [ ]:
```

Simplified additive form. Real RoPE rotates paired channels at frequencies; the positional information is the same.

### Step 3: Divided attention block

In [ ]:
```python

class DividedAttentionBlock(nn.Module):

    def __init__(self, dim=64, heads=2):

        super().__init__()

        self.time_attn = nn.MultiheadAttention(dim, heads, batch_first=True)

        self.space_attn = nn.MultiheadAttention(dim, heads, batch_first=True)

        self.ln1 = nn.LayerNorm(dim)

        self.ln2 = nn.LayerNorm(dim)

        self.ln3 = nn.LayerNorm(dim)

        self.mlp = nn.Sequential(nn.Linear(dim, 4 * dim), nn.GELU(), nn.Linear(4 * dim, dim))

    def forward(self, x, grid):

        T, H, W = grid

        n, seq, d = x.shape

        # time attention: same (h, w), across t

        xt = x.view(n, T, H * W, d).permute(0, 2, 1, 3).reshape(n * H * W, T, d)

        a, _ = self.time_attn(self.ln1(xt), self.ln1(xt), self.ln1(xt), need_weights=False)

        xt = (xt + a).reshape(n, H * W, T, d).permute(0, 2, 1, 3).reshape(n, seq, d)

        # space attention: same t, across (h, w)

        xs = xt.view(n, T, H * W, d).reshape(n * T, H * W, d)

        a, _ = self.space_attn(self.ln2(xs), self.ln2(xs), self.ln2(xs), need_weights=False)

        xs = (xs + a).reshape(n, T, H * W, d).reshape(n, seq, d)

        xs = xs + self.mlp(self.ln3(xs))

        return xs

In [ ]:
```

The time attention attends within each spatial position across time; the space attention attends within each frame across positions. Two O(T^2 + (HW)^2) operations instead of one O((THW)^2). This is the core of TimeSformer and every modern video DiT.

### Step 4: Compose a tiny video DiT

In [ ]:
```python

class TinyVideoDiT(nn.Module):

    def __init__(self, in_channels=4, dim=64, depth=2, heads=2):

        super().__init__()

        self.patch = VideoPatch3D(in_channels=in_channels, dim=dim, patch_t=2, patch_h=2, patch_w=2)

        self.blocks = nn.ModuleList([DividedAttentionBlock(dim, heads) for _ in range(depth)])

        self.out = nn.Linear(dim, in_channels * 2 * 2 * 2)

    def forward(self, x):

        tokens, grid = self.patch(x)

        for blk in self.blocks:

            tokens = blk(tokens, grid)

        return self.out(tokens), grid

In [ ]:
```

Not a working video generator; a structural demo that every piece shapes correctly.

### Step 5: Check shapes

In [ ]:
```python

vid = torch.randn(1, 4, 8, 16, 16)  # (N, C, T, H, W)

model = TinyVideoDiT()

out, grid = model(vid)

print(f"input  {tuple(vid.shape)}")

print(f"tokens grid {grid}")

print(f"output {tuple(out.shape)}")

In [ ]:
```

Expect `grid = (4, 8, 8)` and `out = (1, 256, 32)` after patching; the head then projects to per-token spatio-temporal patches, ready to be un-patchified back into a video.

## Exercises

In [ ]:
1. **(Easy)** Compute the token count for a 5-second 360p video at patch-t=2, patch-h=8, patch-w=8. Reason about memory for attention at this size.
2. **(Medium)** Swap the divided attention block above for a full joint attention block and measure the shape and parameter count. Explain why divided attention is necessary for real video models.
3. **(Hard)** Build a minimal latent-action video model: take a dataset of (frame_t, action_t, frame_{t+1}) triples (any simple 2D game), train a tiny video DiT conditioned on action embeddings, and show that different actions produce different next frames.